# 🤖 Step 7 — FinBERT Sentiment Comparison + 768-dim Embeddings

## What this step does

**Part A — FinBERT vs GPT-4o Comparison (for thesis)**
- Runs FinBERT sentiment on raw GDELT headlines per day
- Compares FinBERT sentiment score vs GPT-4o sentiment score
- Shows which model produces stronger signal

**Part B — FinBERT 768-dim Embeddings (for neural network)**
- Runs FinBERT on GPT-4o daily summaries from Step 6
- Generates a 768-dimensional embedding vector per date
- This is File 2 of your two-file neural network input

## Pipeline position
```
Step 5 raw headlines ──→ FinBERT sentiment  ─→ Compare vs GPT-4o (Part A)
Step 6 GPT-4o summary──→ FinBERT embeddings ─→ 768-dim vectors (Part B)
```

## Inputs
- `step5a_raw_news_all_topics.csv` — raw GDELT headlines
- `step6_gpt4o_daily_summaries.csv` — GPT-4o summaries from Step 6

## Outputs
- `step7a_finbert_sentiment_comparison.csv` — FinBERT vs GPT-4o per date
- `step7b_finbert_embeddings.csv` — 768-dim embeddings per date

---
### Instructions
**Enable GPU first:** Runtime → Change runtime type → T4 GPU → Save

Then run cells in order: **Runtime → Run All**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Install Libraries
# ─────────────────────────────────────────────────────────────────────────
!pip install transformers torch pandas numpy scikit-learn -q
print('✓ Libraries ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Load FinBERT Model
#
# FinBERT (ProsusAI/finbert) is a BERT model fine-tuned on financial text.
# It is hosted on Hugging Face and downloaded for free (~420 MB).
# It runs locally on Colab's GPU — no API calls, no billing.
#
# FinBERT has two uses:
#   1. Sentiment classification → positive / negative / neutral + score
#   2. Embedding generation    → 768-dim vector per text input
# ─────────────────────────────────────────────────────────────────────────
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✓ Device: {device}')
if device == 'cpu':
    print('  ⚠️  No GPU detected — processing will be slower')
    print('  Tip: Runtime → Change runtime type → T4 GPU')

MODEL_NAME = 'ProsusAI/finbert'
print(f'\nLoading FinBERT from Hugging Face...')
print(f'  Model: {MODEL_NAME}')
print(f'  Size : ~420 MB (downloaded once, free)')

# Load tokeniser
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('  ✓ Tokeniser loaded')

# Load sentiment model (for Part A)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
sentiment_model = sentiment_model.to(device)
sentiment_model.eval()
print('  ✓ Sentiment model loaded')

# Load embedding model (for Part B)
embedding_model = AutoModel.from_pretrained(MODEL_NAME)
embedding_model = embedding_model.to(device)
embedding_model.eval()
print('  ✓ Embedding model loaded')

# FinBERT label mapping
LABEL_MAP = {0: 'positive', 1: 'negative', 2: 'neutral'}

print(f'\n✅ FinBERT ready on {device.upper()}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — Upload Input Files
# ─────────────────────────────────────────────────────────────────────────

# Upload Step 5 raw news
print('Upload 1/2: step5a_raw_news_all_topics.csv')
uploaded1  = files.upload()
news_file  = list(uploaded1.keys())[0]
news_df    = pd.read_csv(news_file)
print(f'  ✓ {len(news_df)} articles loaded')
print(f'  Columns: {list(news_df.columns)}')
print(f'  Sample published_at: {news_df["published_at"].iloc[0]}')

# Upload Step 6 GPT-4o summaries
print('\nUpload 2/2: step6_gpt4o_daily_summaries.csv')
uploaded2  = files.upload()
summ_file  = list(uploaded2.keys())[0]
summ_df    = pd.read_csv(summ_file)
print(f'  ✓ {len(summ_df)} daily summaries loaded')
print(f'  Columns: {list(summ_df.columns)}')
print(f'  Sample date: {summ_df["date"].iloc[0]}')

# ── Parse news dates — try multiple formats ────────────────────────────────
def parse_news_date(val):
    """Try multiple date formats from GDELT output."""
    if pd.isna(val) or str(val).strip() == '':
        return None
    val = str(val).strip()
    # Format 1: GDELT full timestamp 20250101T120000Z
    if len(val) >= 15 and val[8] == 'T':
        try:
            return pd.to_datetime(val, format='%Y%m%dT%H%M%SZ').strftime('%Y-%m-%d')
        except:
            pass
    # Format 2: GDELT date only 20250101
    if len(val) == 8 and val.isdigit():
        try:
            return pd.to_datetime(val, format='%Y%m%d').strftime('%Y-%m-%d')
        except:
            pass
    # Format 3: standard datetime or date string
    try:
        return pd.to_datetime(val, errors='coerce').strftime('%Y-%m-%d')
    except:
        return None

# Apply to published_at column
news_df['date'] = news_df['published_at'].apply(parse_news_date)
news_df = news_df[news_df['date'].notna()]
news_df = news_df[news_df['date'] != 'NaT']
news_df['tone'] = pd.to_numeric(news_df['tone'], errors='coerce')

# Also try parsing the 'date' column directly if published_at fails
if len(news_df) == 0 and 'date' in news_df.columns:
    print('  ⚠️  published_at parse failed — trying date column directly')
    news_df = pd.read_csv(news_file)
    news_df['date'] = news_df['date'].apply(parse_news_date)
    news_df = news_df[news_df['date'].notna()]

# ── Parse summary dates ────────────────────────────────────────────────────
summ_df['date_parsed'] = pd.to_datetime(
    summ_df['date'], dayfirst=True, errors='coerce'
).dt.strftime('%Y-%m-%d')
summ_df = summ_df[summ_df['date_parsed'].notna()]
summ_df = summ_df[summ_df['date_parsed'] != 'NaT']
summ_df = summ_df.dropna(subset=['summary'])

news_df['title'] = news_df['title'].astype(str)

print(f'\n  News date range   : {news_df["date"].min()} → {news_df["date"].max()}')
print(f'  News dates        : {news_df["date"].nunique()} unique dates')
print(f'  News date sample  : {sorted(news_df["date"].unique())[:5]}')
print(f'\n  Summary date range: {summ_df["date_parsed"].min()} → {summ_df["date_parsed"].max()}')
print(f'  Summary dates     : {len(summ_df)}')
print(f'  Summary date sample: {sorted(summ_df["date_parsed"].unique())[:5]}')

# Check overlap
overlap = set(news_df['date'].unique()) & set(summ_df['date_parsed'].unique())
print(f'\n  ✓ Overlapping dates: {len(overlap)}')
if len(overlap) == 0:
    print('  ⚠️  Still no overlap — printing raw values for diagnosis:')
    print(f'  Raw news published_at[0]  : {repr(pd.read_csv(news_file)["published_at"].iloc[0])}')
    print(f'  Raw summ date[0]          : {repr(summ_df["date"].iloc[0])}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — Define FinBERT Helper Functions
# ─────────────────────────────────────────────────────────────────────────

def get_finbert_sentiment(text, max_length=512):
    """
    Run FinBERT sentiment classification on a text string.
    Returns:
      label       : positive / negative / neutral
      score       : float -1.0 to +1.0
                    (positive prob - negative prob)
      pos_prob    : probability of positive
      neg_prob    : probability of negative
      neu_prob    : probability of neutral
    """
    if not text or pd.isna(text) or str(text).strip() == '':
        return {'label': 'neutral', 'score': 0.0,
                'pos_prob': 0.0, 'neg_prob': 0.0, 'neu_prob': 1.0}

    inputs = tokenizer(
        str(text),
        return_tensors='pt',
        truncation=True,
        max_length=max_length,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = sentiment_model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1).squeeze().cpu().numpy()
    # FinBERT label order: positive=0, negative=1, neutral=2
    pos_prob = float(probs[0])
    neg_prob = float(probs[1])
    neu_prob = float(probs[2])

    label = LABEL_MAP[int(np.argmax(probs))]
    # Sentiment score: positive - negative (range -1 to +1)
    score = round(pos_prob - neg_prob, 4)

    return {
        'label':    label,
        'score':    score,
        'pos_prob': round(pos_prob, 4),
        'neg_prob': round(neg_prob, 4),
        'neu_prob': round(neu_prob, 4),
    }


def get_finbert_embedding(text, max_length=512):
    """
    Run FinBERT and return a 768-dimensional embedding vector.
    Uses mean pooling across all token embeddings.
    """
    if not text or pd.isna(text) or str(text).strip() == '':
        return np.zeros(768)

    inputs = tokenizer(
        str(text),
        return_tensors='pt',
        truncation=True,
        max_length=max_length,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = embedding_model(**inputs)

    # Mean pool across all token positions
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return embedding


# Quick test
print('Testing FinBERT functions...')
test_sent = get_finbert_sentiment('RBI cuts repo rate boosting market sentiment')
print(f'  Sentiment test : label={test_sent["label"]}  score={test_sent["score"]:+.4f}')
test_emb = get_finbert_embedding('RBI cuts repo rate boosting market sentiment')
print(f'  Embedding test : shape={test_emb.shape}  mean={test_emb.mean():.4f}')
print('\n✅ FinBERT functions ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — PART A: FinBERT Sentiment on Raw Headlines
# ─────────────────────────────────────────────────────────────────────────
import datetime

print('=' * 60)
print('  PART A — FinBERT Sentiment on Raw Headlines')
print('=' * 60)

# Get common dates between news and summaries
news_dates   = set(news_df['date'].unique())
summ_dates   = set(summ_df['date_parsed'].unique())
common_dates = sorted(news_dates & summ_dates)
print(f'  News dates available    : {len(news_dates)}')
print(f'  Summary dates available : {len(summ_dates)}')
print(f'  Common dates            : {len(common_dates)}')

if len(common_dates) == 0:
    print()
    print('  ⚠️  No overlapping dates found')
    print(f'  News dates (sample)   : {sorted(news_dates)[:5]}')
    print(f'  Summary dates (sample): {sorted(summ_dates)[:5]}')
    print()
    print('  Skipping Part A — proceeding to Part B (FinBERT embeddings)')
    part_a_df = pd.DataFrame()
else:
    part_a_results = []

    for i, date in enumerate(common_dates, 1):

        day_news   = news_df[news_df['date'] == date]['title'].dropna().tolist()
        gdelt_tone = news_df[news_df['date'] == date]['tone'].mean()

        if not day_news:
            continue

        daily_scores = []
        daily_pos    = []
        daily_neg    = []
        daily_neu    = []

        for headline in day_news[:30]:
            result = get_finbert_sentiment(headline)
            daily_scores.append(result['score'])
            daily_pos.append(result['pos_prob'])
            daily_neg.append(result['neg_prob'])
            daily_neu.append(result['neu_prob'])

        finbert_score = round(float(np.mean(daily_scores)), 4)
        finbert_label = (
            'positive' if finbert_score > 0.05
            else 'negative' if finbert_score < -0.05
            else 'neutral'
        )

        gpt_row   = summ_df[summ_df['date_parsed'] == date]
        gpt_score = float(gpt_row['sentiment_score'].values[0]) if not gpt_row.empty else np.nan
        gpt_label = str(gpt_row['sentiment'].values[0])          if not gpt_row.empty else ''

        part_a_results.append({
            'date':              pd.to_datetime(date).strftime('%d-%b-%Y'),
            'headline_count':    len(day_news),
            'gdelt_tone':        round(float(gdelt_tone), 4) if pd.notna(gdelt_tone) else None,
            'finbert_score':     finbert_score,
            'finbert_label':     finbert_label,
            'finbert_pos_prob':  round(float(np.mean(daily_pos)), 4),
            'finbert_neg_prob':  round(float(np.mean(daily_neg)), 4),
            'finbert_neu_prob':  round(float(np.mean(daily_neu)), 4),
            'gpt4o_score':       round(gpt_score, 4) if pd.notna(gpt_score) else None,
            'gpt4o_label':       gpt_label,
            'agreement':         finbert_label == gpt_label,
        })

        if i % 50 == 0 or i <= 3:
            print(f'  [{i:>3}/{len(common_dates)}] {date}  '
                  f'FinBERT={finbert_score:+.3f} ({finbert_label:<8})  '
                  f'GPT-4o={gpt_score:+.3f} ({gpt_label})')

    part_a_df = pd.DataFrame(part_a_results)

    print(f'\n✅ Part A complete — {len(part_a_df)} dates processed')
    print(f'\n  Comparison summary:')
    agree_pct = part_a_df['agreement'].mean() * 100
    print(f'    FinBERT vs GPT-4o label agreement : {agree_pct:.1f}%')
    corr = part_a_df['finbert_score'].corr(
        pd.to_numeric(part_a_df['gpt4o_score'], errors='coerce')
    )
    print(f'    Pearson correlation (score)        : {corr:.4f}')
    print(f'\n  FinBERT sentiment distribution:')
    print(part_a_df['finbert_label'].value_counts().to_string())
    print(f'\n  GPT-4o sentiment distribution:')
    print(part_a_df['gpt4o_label'].value_counts().to_string())


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — PART B: FinBERT 768-dim Embeddings on GPT-4o Summaries
#
# For each date:
#   1. Take the GPT-4o generated summary from Step 6
#   2. Run FinBERT encoder on the summary text
#   3. Mean pool token embeddings → 768-dim vector
#   4. This is File 2 for your neural network
#
# Why encode GPT-4o summary instead of raw headlines:
#   - GPT-4o summary is clean, coherent, financial-domain text
#   - FinBERT encodes it better than noisy mixed headlines
#   - Combines GPT-4o understanding + FinBERT financial vocabulary
# ─────────────────────────────────────────────────────────────────────────
print('=' * 60)
print('  PART B — FinBERT 768-dim Embeddings on GPT-4o Summaries')
print('=' * 60)
print(f'  Dates to embed: {len(summ_df)}')
print(f'  Output shape  : ({len(summ_df)}, 768)')
print()

embedding_rows = []
embedding_matrix = []

for i, row in summ_df.iterrows():
    idx = list(summ_df.index).index(i) + 1

    summary = str(row['summary']).strip()
    emb     = get_finbert_embedding(summary)
    embedding_matrix.append(emb)

    embedding_rows.append({
        'date':     row['date'],
        'summary':  summary[:100] + '...' if len(summary) > 100 else summary,
    })

    if idx % 50 == 0 or idx <= 3:
        print(f'  [{idx:>3}/{len(summ_df)}] {row["date"]}  emb_mean={emb.mean():.4f}  emb_std={emb.std():.4f}')

# Build embedding DataFrame
emb_matrix   = np.array(embedding_matrix)
emb_cols     = [f'emb_{i}' for i in range(768)]
emb_df_vals  = pd.DataFrame(emb_matrix, columns=emb_cols)
emb_df_meta  = pd.DataFrame(embedding_rows)
part_b_df    = pd.concat([emb_df_meta.reset_index(drop=True),
                           emb_df_vals.reset_index(drop=True)], axis=1)

print(f'\n✅ Part B complete')
print(f'  Embedding matrix shape : {emb_matrix.shape}')
print(f'  Overall emb mean       : {emb_matrix.mean():.4f}')
print(f'  Overall emb std        : {emb_matrix.std():.4f}')
print(f'  Total columns          : {len(part_b_df.columns)} (2 meta + 768 embedding dims)')

In [ ]:
# Guard against empty Part A results
if part_a_df.empty:
    print('⚠️  Part A has no data — skip to Cell 8 for embeddings only')
else:
    # ─────────────────────────────────────────────────────────────────────────
    # CELL 7 — Comparison Analysis for Thesis
    # ─────────────────────────────────────────────────────────────────────────
    print('=' * 70)
    print('  FINBERT vs GPT-4o SENTIMENT COMPARISON — THESIS ANALYSIS')
    print('=' * 70)
    
    # Agreement analysis
    agree_pct  = part_a_df['agreement'].mean() * 100
    disagree   = part_a_df[~part_a_df['agreement']]
    corr       = part_a_df['finbert_score'].corr(
                     pd.to_numeric(part_a_df['gpt4o_score'], errors='coerce')
                 )
    
    print(f'\n  1. Label Agreement')
    print(f'     FinBERT and GPT-4o agree on sentiment label : {agree_pct:.1f}% of dates')
    print(f'     They disagree on                            : {100-agree_pct:.1f}% of dates ({len(disagree)} dates)')
    
    print(f'\n  2. Score Correlation')
    print(f'     Pearson correlation : {corr:.4f}')
    if corr > 0.7:
            print('     → Strong agreement — both models capture similar market signals')
    elif corr > 0.4:
            print('     → Moderate agreement — GPT-4o adds contextual nuance beyond FinBERT')
    else:
            print('     → Weak agreement — GPT-4o captures different signals than FinBERT alone')
    
    print(f'\n  3. Score Statistics')
    print(f'     {"Metric":<20} {"FinBERT":>12} {"GPT-4o":>12}')
    print('     ' + '─' * 46)
    gpt_scores = pd.to_numeric(part_a_df['gpt4o_score'], errors='coerce')
    fin_scores = part_a_df['finbert_score']
    for metric, fn in [('Mean', np.mean), ('Std', np.std), ('Min', np.min), ('Max', np.max)]:
        print(f'     {metric:<20} {fn(fin_scores):>+12.4f} {fn(gpt_scores.dropna()):>+12.4f}')
    
    print(f'\n  4. Days Where Models Strongly Disagree:')
    part_a_df['score_diff'] = abs(
        part_a_df['finbert_score'] - pd.to_numeric(part_a_df['gpt4o_score'], errors='coerce')
    )
    top_disagree = part_a_df.nlargest(5, 'score_diff')[
        ['date','finbert_score','finbert_label','gpt4o_score','gpt4o_label','score_diff']
    ]
    print(top_disagree.to_string(index=False))
    
    print(f'\n  5. Thesis Insight')
    if corr > 0.5 and agree_pct > 60:
        print('     FinBERT and GPT-4o broadly agree, validating both approaches.')
        print('     GPT-4o adds value through contextual cross-referencing of multiple signals.')
        print('     The combined approach (GPT-4o summary → FinBERT embedding) is expected')
        print('     to outperform either model used in isolation.')
    else:
        print('     Significant divergence between FinBERT and GPT-4o suggests each model')
        print('     captures different dimensions of market sentiment.')
        print('     This supports the multimodal fusion approach — combining both signals')
        print('     in the neural network input should improve predictive accuracy.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 8 — Save and Download Both Output Files
# ─────────────────────────────────────────────────────────────────────────
import os

# ── Save Part A — Comparison file ─────────────────────────────────────────
FILE_A = 'step7a_finbert_sentiment_comparison.csv'
part_a_df.to_csv(FILE_A, index=False)
size_a = os.path.getsize(FILE_A)

# ── Save Part B — Embeddings file ─────────────────────────────────────────
FILE_B = 'step7b_finbert_embeddings.csv'
# Save only date + 768 embedding columns (drop summary preview)
save_b = part_b_df.drop(columns=['summary'], errors='ignore')
save_b.to_csv(FILE_B, index=False)
size_b = os.path.getsize(FILE_B)

print('=' * 60)
print('  FILES SAVED')
print('=' * 60)

print(f'\n  Part A — FinBERT vs GPT-4o Comparison:')
print(f'    File    : {FILE_A}')
print(f'    Rows    : {len(part_a_df)} dates')
print(f'    Columns : {len(part_a_df.columns)}')
print(f'    Size    : {size_a/1024:.1f} KB')
print(f'    Columns :')
col_desc_a = {
    'date':             'Trading date',
    'headline_count':   'Number of headlines processed',
    'gdelt_tone':       'Raw GDELT tone score',
    'finbert_score':    'FinBERT sentiment score (-1 to +1)',
    'finbert_label':    'FinBERT label: positive/negative/neutral',
    'finbert_pos_prob': 'FinBERT avg positive probability',
    'finbert_neg_prob': 'FinBERT avg negative probability',
    'finbert_neu_prob': 'FinBERT avg neutral probability',
    'gpt4o_score':      'GPT-4o sentiment score (-1 to +1)',
    'gpt4o_label':      'GPT-4o label: bullish/bearish/neutral',
    'agreement':        'True if both models agree on label',
    'score_diff':       'Absolute difference between scores',
}
for col in part_a_df.columns:
    print(f'      {col:<22} — {col_desc_a.get(col, "")}')

print(f'\n  Part B — FinBERT Embeddings:')
print(f'    File    : {FILE_B}')
print(f'    Rows    : {len(save_b)} dates')
print(f'    Columns : date + 768 embedding dimensions = {len(save_b.columns)}')
print(f'    Size    : {size_b/1024/1024:.1f} MB')
print(f'    Use     : Feed emb_0 ... emb_767 into neural network (Step 9)')

print()
print('⬇️  Downloading files...')
files.download(FILE_A)
print(f'  ✓ {FILE_A}')
files.download(FILE_B)
print(f'  ✓ {FILE_B}')

print()
print('✅ Step 7 Complete!')
print()
print('  Step 8 — Two feature files are now ready:')
print('  File 1 → Normalised numerical features  (from Step 4)')
print('           OHLC log returns + EMA + RSI + BB + MACD')
print('  File 2 → FinBERT embeddings              (step7b_finbert_embeddings.csv)')
print('           768-dim vector per date')
print()
print('  Next → Step 9: Feed both files into Neural Network')